# Notebook 3: finish the GPU work (NLLB en-mni + COMET for all systems)
Needs the bundle built **with outputs**: `python scripts/make_bundle.py --with-outputs` (adds every `test.hyp`, no checkpoints).

**Colab A100** (recommended): put `nmt_bundle.zip` in My Drive, Runtime, then A100, then Run all. About 25 min.
**Kaggle T4:** works too; the en-mni run then uses batch 4 × accumulation 8 automatically. About 1.5 h.

Result: `nb3_results.zip` (My Drive root / Kaggle Output). Extract it in the project root, then run
`bash run.sh eval && bash run.sh figures && bash run.sh presentation`.

In [1]:
# ---- 1. Locate the project bundle and unpack it (works on Kaggle and Colab) ----
import os, sys, glob, zipfile, shutil, subprocess
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
# NB: recent Colab images also contain a /kaggle folder, so check Colab first
IN_KAGGLE = (not IN_COLAB) and ("KAGGLE_KERNEL_RUN_TYPE" in os.environ or os.path.exists("/kaggle/input"))
WORK = "/kaggle/working/project" if IN_KAGGLE else "/content/project"

def find_bundle():
    pats = ["/kaggle/input/**/nmt_bundle.zip", "/content/nmt_bundle.zip",
            "/content/drive/MyDrive/nmt_bundle.zip", "/content/drive/MyDrive/**/nmt_bundle.zip"]
    for p in pats:
        hits = glob.glob(p, recursive=True)
        if hits:
            return ("zip", hits[0])
    # Kaggle auto-extracts uploaded zips -> look for an extracted project folder
    for d in glob.glob("/kaggle/input/**/src/models", recursive=True):
        return ("dir", os.path.dirname(os.path.dirname(d)))
    return (None, None)

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

kind, path = find_bundle()
if kind is None and IN_COLAB:
    from google.colab import files
    print("Upload dist/nmt_bundle.zip"); up = files.upload(); kind, path = "zip", "/content/" + list(up)[0]
assert kind, "nmt_bundle.zip not found - attach it as a Kaggle dataset / put it in MyDrive"
if not os.path.exists(WORK + "/src"):
    os.makedirs(WORK, exist_ok=True)
    if kind == "zip":
        zipfile.ZipFile(path).extractall(WORK)
    else:
        shutil.copytree(path, WORK, dirs_exist_ok=True)
os.chdir(WORK); sys.path.insert(0, WORK)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # less fragmentation (inherited by jobs)

# On Colab keep outputs on Google Drive so a disconnect never loses finished runs
if IN_COLAB:
    drive_out = "/content/drive/MyDrive/nmt_outputs"
    os.makedirs(drive_out, exist_ok=True)
    if not os.path.islink("outputs"):
        if os.path.isdir("outputs"):
            shutil.copytree("outputs", drive_out, dirs_exist_ok=True); shutil.rmtree("outputs")
        os.symlink(drive_out, "outputs")
print("bundle:", path, "| workdir:", os.getcwd())

Mounted at /content/drive
bundle: /content/drive/MyDrive/dist/nmt_bundle.zip | workdir: /content/project


In [2]:
# ---- 2. Dependencies + GPU check ----
!pip install -q -r requirements-gpu.txt
import torch
n = torch.cuda.device_count()
print("torch", torch.__version__, "| GPUs:", n, [torch.cuda.get_device_name(i) for i in range(n)])
print("bf16 supported:", torch.cuda.is_bf16_supported() if n else None)
assert n > 0, "No GPU! Kaggle: Settings -> Accelerator -> GPU T4 x2.  Colab: Runtime -> Change runtime type -> A100"

torch 2.14.1+cu130 | GPUs: 1 ['NVIDIA L4']
bf16 supported: True


In [3]:
# ---- helpers: skip finished runs (safe to re-run the notebook after a disconnect) ----
import json
from pathlib import Path
from src.run_queue import run_jobs

def finished(model, direction):
    f = Path("outputs") / model / direction / "run_config.json"
    return f.exists() and json.loads(f.read_text()).get("finished", False)

def todo(jobs):
    return [(n, a) for (n, a, m, d) in jobs if not finished(m, d)]

DIRECTIONS = ["en-as", "as-en", "en-lus", "lus-en", "en-mni", "mni-en", "en-bodo", "bodo-en"]  # big pairs first

In [4]:
# ---- 3. NLLB en->mni (the run that ran out of memory on the T4). Skipped if already finished ----
d = "en-mni"
if not finished("nllb_finetune", d):
    extra = [] if torch.cuda.get_device_properties(0).total_memory > 30e9 else ["--batch-size", "4", "--grad-accum", "8"]
    print(run_jobs([(f"nllb_{d}", ["-m", "src.models.advanced.nllb_finetune", "--direction", d] + extra)]))
r = json.loads(Path(f"outputs/nllb_finetune/{d}/run_config.json").read_text())
print(f"{d}: zero-shot dev chrF++ {r['zeroshot_dev_chrf']:.2f} -> fine-tuned {r['dev_chrf_beam']:.2f}  ({r['wall_clock_s']/60:.0f} min)")

en-mni: zero-shot dev chrF++ 40.73 -> fine-tuned 55.88  (60 min)


In [5]:
# ---- COMET (optional; installed last so it cannot break the training environment) ----
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "unbabel-comet"], capture_output=True, text=True)
print("pip:", "OK" if r.returncode == 0 else r.stderr[-3000:])
env = dict(os.environ, USE_TF="0", TRANSFORMERS_NO_TF="1")
r = subprocess.run([sys.executable, "-m", "src.eval.comet_score", "--models"] + ["baseline", "backtranslation", "nllb_finetune"], capture_output=True, text=True, env=env)
print(r.stdout[-3000:])
if r.returncode != 0:
    print("COMET failed (non-fatal; chrF++/BLEU are computed locally). Error tail:")
    print(r.stderr[-4000:])

pip: OK
COMET backtranslation/as-en: 0.6331
COMET backtranslation/bodo-en: 0.6828
COMET backtranslation/en-as: 0.6144
COMET backtranslation/en-bodo: 0.6236
COMET backtranslation/en-lus: 0.6611
COMET backtranslation/en-mni: 0.6925
COMET backtranslation/lus-en: 0.5639
COMET backtranslation/mni-en: 0.6563
COMET baseline/as-en: 0.6287
COMET baseline/bodo-en: 0.6631
COMET baseline/en-as: 0.6510
COMET baseline/en-bodo: 0.6255
COMET baseline/en-lus: 0.6675
COMET baseline/en-mni: 0.6964
COMET baseline/lus-en: 0.5656
COMET baseline/mni-en: 0.6692
COMET nllb_finetune/as-en: 0.7870
COMET nllb_finetune/bodo-en: 0.7801
COMET nllb_finetune/en-as: 0.7907
COMET nllb_finetune/en-bodo: 0.6472
COMET nllb_finetune/en-lus: 0.7000
COMET nllb_finetune/en-mni: 0.7558
COMET nllb_finetune/lus-en: 0.6846
COMET nllb_finetune/mni-en: 0.8011



In [6]:
# ---- 5. Package: en-mni NLLB outputs + every comet.json -> MyDrive/nb3_results.zip (or Kaggle Output) ----
import zipfile
out_dir = "/kaggle/working" if IN_KAGGLE else "/content/drive/MyDrive"
res = f"{out_dir}/nb3_results.zip"
with zipfile.ZipFile(res, "w", zipfile.ZIP_DEFLATED) as z:
    for f in list(Path("outputs/nllb_finetune/en-mni").glob("*")) + list(Path("outputs").glob("*/*/comet.json")) \
             + list(Path("outputs/logs").glob("nllb_en-mni.log")):
        if f.is_file():
            z.write(f, f.as_posix()); print("  +", f)
print(res, round(os.path.getsize(res) / 1e6, 2), "MB  -> extract in the project root")

  + outputs/nllb_finetune/en-mni/zeroshot_dev.hyp
  + outputs/nllb_finetune/en-mni/train_log.csv
  + outputs/nllb_finetune/en-mni/dev.hyp
  + outputs/nllb_finetune/en-mni/test.hyp
  + outputs/nllb_finetune/en-mni/run_config.json
  + outputs/nllb_finetune/en-mni/comet.json
  + outputs/baseline/en-as/comet.json
  + outputs/baseline/as-en/comet.json
  + outputs/baseline/en-lus/comet.json
  + outputs/baseline/lus-en/comet.json
  + outputs/baseline/en-mni/comet.json
  + outputs/baseline/mni-en/comet.json
  + outputs/baseline/en-bodo/comet.json
  + outputs/baseline/bodo-en/comet.json
  + outputs/backtranslation/en-as/comet.json
  + outputs/backtranslation/as-en/comet.json
  + outputs/backtranslation/en-lus/comet.json
  + outputs/backtranslation/lus-en/comet.json
  + outputs/backtranslation/en-mni/comet.json
  + outputs/backtranslation/mni-en/comet.json
  + outputs/backtranslation/en-bodo/comet.json
  + outputs/backtranslation/bodo-en/comet.json
  + outputs/nllb_finetune/mni-en/comet.json
  +

/usr/lib/python3.13/zipfile/__init__.py:1686: UserWarning: Duplicate name: 'outputs/nllb_finetune/en-mni/comet.json'
  return self._open_to_write(zinfo, force_zip64=force_zip64)
